In [1]:
import json
from helper import helpers as hp
import pandas as pd
from tqdm import tqdm

In [2]:
import importlib

# Data Loading

In [3]:
kpi_concept_meta_root = "./data/concept/"
respondent_level_meta_root = "./data/respondent/"
local_data_root = "./data/"

In [4]:
# get the concept file
with open(kpi_concept_meta_root + "cid_concept_us_food.json") as f:
    cid_concept = json.load(f)

# rephrasing

In [6]:
import importlib
from augmentation import rephrasing
importlib.reload(rephrasing)


from augmentation.rephrasing import ConceptRephraser, RephraseConfig, Tone, PointOfView, ContentOrder

# Initialize the rephraser with the API key from the module
rephraser = ConceptRephraser(model="gpt-4o-mini", temperature=0.5)

c:\Users\Yuding.Duan\OneDrive - Ipsos\3. self_projects\llm_synthetic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import random

def multi_rephrase_concept(concept_text, n_variations=8):
    """
    Generate n random variations of a concept using different rephrasing configurations.
    
    Args:
        concept_text: The original concept text
        n_variations: Number of variations to generate (default: 5)
    
    Returns:
        List of dicts with 'config' and 'text' keys
    """
    tones = [None, Tone.CLINICAL, Tone.MARKETING, Tone.CONVERSATIONAL]
    povs = [None, PointOfView.SECOND_PERSON, PointOfView.THIRD_PERSON]
    orders = [None, ContentOrder.PROBLEM_FIRST, ContentOrder.FEATURE_FIRST, ContentOrder.BENEFIT_FIRST]
    length_options = [True, False]
    
    # Generate unique random configurations
    seen_configs = set()
    variations = []
    
    while len(variations) < n_variations:
        config_tuple = (
            random.choice(tones),
            random.choice(povs),
            random.choice(orders),
            random.choice(length_options)
        )
        
        # Skip if we've already used this exact config
        if config_tuple in seen_configs:
            continue
        seen_configs.add(config_tuple)
        
        tone, pov, order, change_len = config_tuple
        
        config = RephraseConfig(
            change_length=change_len,
            tone=tone,
            point_of_view=pov,
            content_order=order,
        )
        
        result = rephraser.rephrase(concept_text, config)
        
        variations.append({
            'config': {
                'tone': tone.value if tone else None,
                'point_of_view': pov.value if pov else None,
                'content_order': order.value if order else None,
                'change_length': change_len,
            },
            'text': result.rephrased_text,
            'word_count': result.new_word_count,
        })
    
    return variations


# Test it
sample_cid = list(cid_concept.keys())[0]
sample_text = cid_concept[sample_cid]
print(f"Original ({len(sample_text.split())} words):\n{sample_text}\n")
print("=" * 60)

variations = multi_rephrase_concept(sample_text, n_variations=8)

for i, var in enumerate(variations, 1):
    print(f"\n[Variation {i}] Config: {var['config']}")
    print(f"Word count: {var['word_count']}")
    print(var['text'])
    print("-" * 60)

Original (89 words):
OIKOS TRIPLE ZERO PLAIN HIGH PROTEIN YOGURT

More of what you want, less of what you don't - 18g of protein 0% fat, 0g added sugars and 0 artificial sweeteners.

Available in 32oz large size, Oikos Triple Zero Plain is a deliciously simple way to get the protein you need. Perfect to add to smoothies, parfaits, or enjoy on it's own!

- 18g Protein
- 0g Added Sugar
- 0 Artificial Sweeteners
- 0% Fat
- Project Non-GMO Verified

32oz Multi-Serve Tub - $5.99

Current Oikos Assortment Still Available



KeyboardInterrupt: 

In [9]:
from tqdm import tqdm

variation_cid_concept = {}

for k, v in tqdm(cid_concept.items(), desc="Processing concepts"):
    variations = multi_rephrase_concept(v, n_variations=8)
    sub_variations = {i+1: var for i, var in enumerate(variations)}
    variation_cid_concept[k] = sub_variations

Processing concepts:   0%|          | 0/12 [00:00<?, ?it/s]

Processing concepts: 100%|██████████| 12/12 [05:25<00:00, 27.13s/it]


In [12]:
with open(local_data_root + "0414_variation_cid_concept.json", 'w') as f:
    json.dump(variation_cid_concept, f, indent=2)